# BiomedicalDataset Visualization & Testing

This notebook tests and visualizes the BiomedicalDataset for ViT finetuning.

**Tests:**
- Dataset loading from Hugging Face datasets
- Train/val split
- Preprocessing (min-max normalization, upsampling)
- DataLoader batching
- Image visualization

In [ ]:
# Imports
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

from orochi.configs.model_configs import ViT3DConfig
from src.finetune_with_wandb import BiomedicalDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Dataset Configuration

In [ ]:
# Configuration
config = ViT3DConfig()

print("Dataset Configuration:")
print(f"  Image size: {config.img_size}")
print(f"  Batch size: {config.batch_size}")
print(f"  Patch size: {config.patch_size}")
print(f"  Embed dim: {config.embed_dim}")

## 2. Create Datasets

In [ ]:
# Create train and val datasets
train_dataset = BiomedicalDataset(
    data_root="/group/jug/aman/orochi/data",
    datasets=['hipsc_3d', 'hipsc_2d', 'hipct_2d', 'idr_2d'],
    img_size=config.img_size,
    split='train',
    val_split=0.1
)

val_dataset = BiomedicalDataset(
    data_root="/group/jug/aman/orochi/data",
    datasets=['hipsc_3d', 'hipsc_2d', 'hipct_2d', 'idr_2d'],
    img_size=config.img_size,
    split='val',
    val_split=0.1
)

print(f"\n✓ Total samples: {len(train_dataset) + len(val_dataset)}")

## 3. Inspect Individual Samples

In [ ]:
# Load first sample
sample = train_dataset[0]
image = sample['image']
idx = sample['idx']

print(f"Sample {idx}:")
print(f"  Shape: {image.shape}")
print(f"  Dtype: {image.dtype}")
print(f"  Range: [{image.min():.4f}, {image.max():.4f}]")
print(f"  Mean: {image.mean():.4f}")
print(f"  Std: {image.std():.4f}")

## 4. Visualization Functions

In [ ]:
def visualize_mid_slice(volume, title="Mid Slice", cmap='gray'):
    """Visualize middle slice of 3D volume."""
    if volume.ndim == 4:  # (C, D, H, W)
        mid_slice = volume[0, volume.shape[1] // 2]
    elif volume.ndim == 3:  # (D, H, W)
        mid_slice = volume[volume.shape[0] // 2]
    else:
        raise ValueError(f"Expected 3D or 4D volume, got {volume.ndim}D")
    
    # Convert to numpy if needed
    if torch.is_tensor(mid_slice):
        mid_slice = mid_slice.cpu().numpy()
    
    plt.figure(figsize=(8, 8))
    plt.imshow(mid_slice, cmap=cmap)
    plt.title(f"{title}\nShape: {mid_slice.shape}, Range: [{mid_slice.min():.3f}, {mid_slice.max():.3f}]")
    plt.colorbar()
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def visualize_3d_montage(volume, num_slices=8, title="3D Volume", cmap='gray'):
    """Visualize multiple slices from 3D volume."""
    if volume.ndim == 4:  # (C, D, H, W)
        volume = volume[0]  # Take first channel
    
    if torch.is_tensor(volume):
        volume = volume.cpu().numpy()
    
    D = volume.shape[0]
    indices = np.linspace(0, D-1, num_slices, dtype=int)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        axes[i].imshow(volume[idx], cmap=cmap)
        axes[i].set_title(f"Slice {idx}/{D}")
        axes[i].axis('off')
    
    plt.suptitle(f"{title}\nShape: {volume.shape}", fontsize=14)
    plt.tight_layout()
    plt.show()


def visualize_batch(images, num_samples=4, title="Batch Samples"):
    """Visualize multiple samples from a batch."""
    num_samples = min(num_samples, images.shape[0])
    
    fig, axes = plt.subplots(1, num_samples, figsize=(4*num_samples, 4))
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        # Get middle slice of each sample
        img = images[i]
        mid_slice = img[0, img.shape[1] // 2].cpu().numpy()
        
        axes[i].imshow(mid_slice, cmap='gray')
        axes[i].set_title(f"Sample {i}\nRange: [{mid_slice.min():.2f}, {mid_slice.max():.2f}]")
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

print("✓ Visualization functions defined")

## 5. Visualize Single Sample

In [ ]:
# Visualize middle slice
visualize_mid_slice(image, title=f"Sample {idx} - Mid Slice")

In [ ]:
# Visualize 3D montage
visualize_3d_montage(image, num_slices=8, title=f"Sample {idx} - 3D Montage")

## 6. Test DataLoader

In [ ]:
# Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"DataLoader created:")
print(f"  Batch size: 4")
print(f"  Total batches: {len(train_loader)}")

# Get first batch
batch = next(iter(train_loader))
images = batch['image']
indices = batch['idx']

print(f"\nFirst batch:")
print(f"  Batch shape: {images.shape}")
print(f"  Indices: {indices.tolist()}")
print(f"  Memory: {images.element_size() * images.nelement() / 1024**2:.2f} MB")

In [ ]:
# Visualize batch
visualize_batch(images, num_samples=4, title="First Batch - Mid Slices")

## 7. Data Distribution Analysis

In [ ]:
# Analyze intensity distributions
print("Analyzing intensity distributions (first 10 samples)...\n")

stats = []
for i in range(min(10, len(train_dataset))):
    sample = train_dataset[i]
    img = sample['image']
    
    stats.append({
        'min': img.min().item(),
        'max': img.max().item(),
        'mean': img.mean().item(),
        'std': img.std().item()
    })

# Display stats
for i, s in enumerate(stats):
    print(f"Sample {i}: min={s['min']:.4f}, max={s['max']:.4f}, mean={s['mean']:.4f}, std={s['std']:.4f}")

# Plot histogram of means
means = [s['mean'] for s in stats]
plt.figure(figsize=(10, 4))
plt.hist(means, bins=20)
plt.xlabel('Mean Intensity')
plt.ylabel('Count')
plt.title('Distribution of Mean Intensities (First 10 Samples)')
plt.grid(True, alpha=0.3)
plt.show()

## 8. File Distribution Across Datasets

In [ ]:
# Count files per dataset
file_counts = {}
for img_file in train_dataset.image_files:
    # Extract dataset name from path
    if 'data' in img_file.parts:
        dataset_name = img_file.parts[-3]  # Parent of 'data'
    else:
        dataset_name = img_file.parts[-2]  # Parent directory
    
    file_counts[dataset_name] = file_counts.get(dataset_name, 0) + 1

# Plot distribution
datasets = list(file_counts.keys())
counts = list(file_counts.values())

plt.figure(figsize=(10, 6))
plt.bar(datasets, counts)
plt.xlabel('Dataset')
plt.ylabel('Number of Files')
plt.title('Training Set File Distribution')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nFile counts:")
for ds, count in sorted(file_counts.items()):
    print(f"  {ds}: {count} files")

## 9. Summary

In [ ]:
print("=" * 60)
print("Dataset Test Summary")
print("=" * 60)
print(f"\n✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")
print(f"✓ Image size: {config.img_size}")
print(f"✓ Preprocessing: min-max normalization")
print(f"✓ DataLoader: batching works correctly")
print(f"\n✓ All tests passed! Ready for training.")
print(f"\nNext step:")
print(f"  python src/finetune_with_wandb.py --config configs/vit_finetune.yaml --model vit")